In [3]:
import pandas as pd
import json
from langgraph_graph import graph, collector
from observability_db import init_db, save_run, get_run
from langchain.messages import HumanMessage
import requests

In [4]:
init_db()

#### 1. Using python functions

In [5]:
df = pd.read_excel("input_file.xlsx")
print(df.shape)
df.head()

(5, 1)


,input
0,what is 3 multiplied by 77?
1,what is 5 multiplied by 88?
2,what is 10 multiplied by 60?
3,what is 4 multiplied by 101?
4,what is 55 multiplied by 454?


In [6]:
for i, row in df.iterrows():
    print(i)

    user_input = row["input"]

    collector.reset()

    # Run graph
    result = graph.invoke(
        {"messages": [HumanMessage(content=user_input)]}
    )

    trace_json = collector.build()

    # attach user query
    trace_json["user_query"] = user_input
    trace_json["final_response"] = result["messages"][-1].content

    # Save to DB
    save_run(**trace_json)

    # Get run_id from saved trace
    new_run_id = trace_json["run_id"]

    df.loc[i, "final_response"] = result["messages"][-1].content
    df.loc[i, "run_id"] = new_run_id

print("done!!")
df

0
1
2
3
4
done!!


,input,final_response,run_id
0,what is 3 multiplied by 77?,The result of 3 multiplied by 77 is 231.,799fc0d4-f5de-41b7-92be-8761106509bf
1,what is 5 multiplied by 88?,The result of 5 multiplied by 88 is 440.,8041f9cd-21a7-44ca-b20e-6dd51948af5e
2,what is 10 multiplied by 60?,The result of 10 multiplied by 60 is 600.,07113411-af2e-42cc-9ed4-820ac671f451
3,what is 4 multiplied by 101?,The result of 4 multiplied by 101 is 404.,84105558-0386-4517-86d8-0c42b7c3e87a
4,what is 55 multiplied by 454?,The result of 55 multiplied by 454 is 24970.,3658fa9f-3218-40cb-8a69-cf5b6ac55d99


In [7]:
run_id = df.loc[3, "run_id"]
print(json.dumps(get_run(run_id), indent=4))

{
    "run_id": "84105558-0386-4517-86d8-0c42b7c3e87a",
    "user_query": "what is 4 multiplied by 101?",
    "final_response": "The result of 4 multiplied by 101 is 404.",
    "start_time": "03 Mar 2026, 07:55:04.378 PM",
    "start_timestamp": "2026-03-03T19:55:04.378309",
    "total_tokens": {
        "input_tokens": 962,
        "output_tokens": 33,
        "total_tokens": 995
    },
    "total_latency": 0.385,
    "errors": [],
    "nodes": [
        {
            "node": "llm_call",
            "start": "03 Mar 2026, 07:55:04.379 PM",
            "latency": 0.195,
            "tokens": {
                "input_tokens": 466,
                "output_tokens": 19,
                "total_tokens": 485
            },
            "input": "================================ Human Message =================================\n\nwhat is 4 multiplied by 101?\n\n",
            "output": "================================== Ai Message ==================================\nTool Calls:\n  multiply (nes

#### 2. Using running app

In [8]:
df = pd.read_excel("input_file.xlsx")
print(df.shape)
df.head()

(5, 1)


,input
0,what is 3 multiplied by 77?
1,what is 5 multiplied by 88?
2,what is 10 multiplied by 60?
3,what is 4 multiplied by 101?
4,what is 55 multiplied by 454?


In [9]:
url = "http://127.0.0.1:5000/batchrun"

payload = {
    "message": "Hello API"
}

response = requests.post(url, data=payload)

print("STATUS CODE:")
print(response.status_code)
print("RESPONSE JSON:")
print(json.dumps(response.json(), indent=4))
print("OUTPUT URL:")
print(f"http://127.0.0.1:5000/?run_id={response.json().get('run_id')}")

STATUS CODE:
200
RESPONSE JSON:
{
    "errors": [],
    "final_response": "I'm here to help with arithmetic operations. What would you like to calculate?",
    "nodes": [
        {
            "error": null,
            "input": "================================ Human Message =================================\n\nHello API\n\n",
            "latency": 0.272,
            "node": "llm_call",
            "output": "================================== Ai Message ==================================\n\nI'm here to help with arithmetic operations. What would you like to calculate?",
            "start": "03 Mar 2026, 07:56:31.890 PM",
            "timestamp": "2026-03-03T19:56:32.161967",
            "tokens": {
                "input_tokens": 459,
                "output_tokens": 17,
                "total_tokens": 476
            }
        }
    ],
    "run_id": "4b561148-f484-4fd2-b02d-57c9c06d4401",
    "start_time": "03 Mar 2026, 07:56:31.888 PM",
    "start_timestamp": "2026-03-03T19:56:31

In [10]:
url = "http://127.0.0.1:5000/batchrun"
for i, row in df.iterrows():
    print(i)

    payload = {"message": row["input"]}
    response = requests.post(url, data=payload)
    run_id = response.json().get('run_id')
    final_response = response.json().get('final_response')

    df.loc[i, "final_response"] = final_response
    df.loc[i, "output_url"] = f"http://127.0.0.1:5000/?run_id={run_id}"

print("done!!")
df

0
1
2
3
4
done!!


,input,final_response,output_url
0,what is 3 multiplied by 77?,The result of 3 multiplied by 77 is 231.,http://127.0.0.1:5000/?run_id=35d95082-c2a9-49...
1,what is 5 multiplied by 88?,The result of 5 multiplied by 88 is 440.,http://127.0.0.1:5000/?run_id=aaddf8b2-dfff-49...
2,what is 10 multiplied by 60?,The result of 10 multiplied by 60 is 600.,http://127.0.0.1:5000/?run_id=09e0fdb7-869f-41...
3,what is 4 multiplied by 101?,The result of 4 multiplied by 101 is 404.,http://127.0.0.1:5000/?run_id=3681ff37-b582-49...
4,what is 55 multiplied by 454?,The result of 55 multiplied by 454 is 24970.,http://127.0.0.1:5000/?run_id=bfcf8abe-e442-43...


In [11]:
df.loc[0, "output_url"]

'http://127.0.0.1:5000/?run_id=35d95082-c2a9-49db-b483-838d1ecc7940'